In [2]:
# Libraries
import numpy as np
import pandas as pd
import os
import requests
from matplotlib.pylab import array
import goatools

In [3]:
# Getting sampled Parquet data
# CURRENT USER MUST CHANGE THE PATH TO THE PARQUET FILE TO THEIR OWN PATH!!!!!!!!!!
parquet_data = pd.read_parquet(r'C:\Users\ashto\ddi-prediction\data\sample\randomly_sampled_1000_drugs.parquet')

print(parquet_data.head())

#print(parquet_data.columns)
parquet_target_data = parquet_data[['drug/drugbank-id', 'drug/external_identifiers', 'drug/name', 'drug/n_targets', 'drug/targets']]

#print(parquet_target_data.head())

  drug/drugbank-id     drug[@type]  drug/n_interactions  \
0          DB02351  small molecule                  641   
1        APRD00718  small molecule                 1294   
2        BIOD00036  small molecule                   56   
3          DB12161  small molecule                 1517   
4          DB11575  small molecule                  435   

                                   drug/interactions  drug/n_targets  \
0  [{'description': 'Apixaban may increase the an...               1   
1  [{'description': 'Chlorprothixene may increase...              11   
2  [{'description': 'The therapeutic efficacy of ...               1   
3  [{'description': 'The risk or severity of adve...               1   
4  [{'description': 'The serum concentration of G...               2   

                                        drug/targets  drug/n_atc_codes  \
0  [{'actions': ['inhibitor'], 'id': 'BE0000048',...                 1   
1  [{'actions': ['antagonist'], 'id': 'BE0000756'...            

In [3]:
print(parquet_data.columns)

Index(['drug/drugbank-id', 'drug[@type]', 'drug/n_interactions',
       'drug/interactions', 'drug/n_targets', 'drug/targets',
       'drug/n_atc_codes', 'drug/atc_codes', 'drug/n_reactions',
       'drug/reactions', 'drug/external_identifiers', 'drug/n_groups',
       'drug/groups', 'drug/drugbank-id[@primary]', 'drug/name',
       'drug/cas-number', 'drug/unii', 'drug/average-mass',
       'drug/monoisotopic-mass',
       'drug/calculated-properties/property[8]/property/kind',
       'drug/calculated-properties/property[8]/property/value',
       'drug/calculated-properties/property[8]/property/source',
       'drug/calculated-properties/property[11]/property/kind',
       'drug/calculated-properties/property[11]/property/value',
       'drug/calculated-properties/property[11]/property/source',
       'drug/calculated-properties/property[5]/property/kind',
       'drug/calculated-properties/property[5]/property/value',
       'drug/calculated-properties/property[5]/property/source',


In [3]:
# Target Identifiers
# These are saverd as a nested structure of numpy arrrays and dicts
# Structure: array --> dict --> fields (some of which are arrays themselves) --> dict --> Fields

from matplotlib.pylab import array

parquet_target_ids = parquet_target_data[['drug/name', 'drug/targets']]
print(parquet_target_ids.head())

          drug/name                                       drug/targets
0       Bivalirudin  [{'actions': ['inhibitor'], 'id': 'BE0000048',...
1   Chlorprothixene  [{'actions': ['antagonist'], 'id': 'BE0000756'...
2      Gramicidin D  [{'actions': ['binder'], 'id': 'BE0000803', 'k...
3  Deutetrabenazine  [{'actions': ['inhibitor'], 'id': 'BE0000118',...
4       Grazoprevir  [{'actions': ['antagonist'], 'id': 'BE0004701'...


In [4]:

# Step 1: Extract unique UniProt IDs and drug-target mappings from the parquet data

def extract_target_ids(parquet_of_target_ids):
    count_num_target_not_poly = 0
    """Extract drug-target-polypeptide mappings into a flat DataFrame."""
    rows = []
    for _, row in parquet_of_target_ids.iterrows(): # Iterate over each drug (row not index)
        if row['drug/targets'] is None: # If no targets, skip this drug
            continue
        for target in row['drug/targets']: # Iterate over each target of the drug
            if target.get('polypeptides') is None: # If no polypeptides, skip this target 
                count_num_target_not_poly += 1
                continue
            for poly in target['polypeptides']: # Iterate over each polypeptide of the target and extract relevant info
                rows.append({
                    'drug': row['drug/name'],
                    'target': target['name'],
                    'uniprot_id': poly['ext_ids']['UniProtKB'],
                    'gene': poly['gene-name'],
                    'source': poly['source']
                })
    print(f"Number of targets without polypeptides: {count_num_target_not_poly} JUST INTERESTED IN THIS NUMBER")
    return pd.DataFrame(rows) # Return a DataFrame with columns: drug, target, uniprot_id, gene, source

target_df = extract_target_ids(parquet_target_ids) # Extract the drug-target-polypeptide mappings into a flat DataFrame
print(f"Total mappings: {len(target_df)}")
print(f"Unique UniProt IDs: {target_df['uniprot_id'].nunique()}") # Print the number of unique UniProt IDs in the extracted DataFrame
target_df.head()


Number of targets without polypeptides: 0 JUST INTERESTED IN THIS NUMBER
Total mappings: 3323
Unique UniProt IDs: 1009


,drug,target,uniprot_id,gene,source
0,Bivalirudin,Prothrombin,P00734,F2,Swiss-Prot
1,Chlorprothixene,D(2) dopamine receptor,P14416,DRD2,Swiss-Prot
2,Chlorprothixene,D(1A) dopamine receptor,P21728,DRD1,Swiss-Prot
3,Chlorprothixene,D(3) dopamine receptor,P35462,DRD3,Swiss-Prot
4,Chlorprothixene,5-hydroxytryptamine receptor 2A,P28223,HTR2A,Swiss-Prot


In [5]:
# Step 2: Batch-fetch FASTA sequences (with isoforms) from UniProt REST API

def fetch_fasta_batch(uniprot_ids, include_isoforms=True, batch_size=100):
    """
    Fetch FASTA sequences for a list of UniProt IDs using the UniProt REST API.
    Batches requests to avoid overly long query strings.
    Returns a dict: {uniprot_id: fasta_string}. Including isoforms is currertly default True, 
    but can be set to False if only canonical sequences are desired. 
    Isoforms will be grouped under their canonical ID (e.g. P00734-2 will be grouped under P00734).
    """
    unique_ids = list(set(uniprot_ids)) # Get unique IDs to avoid redundant API calls using set() to remove duplicates and then convert back to list
    results = {} # Store results as {fasta_id: fasta_string} where fasta_id can be canonical or isoform (e.g. P00734 or P00734-2)
    session = requests.Session() # Use a session for connection pooling and performance

    for i in range(0, len(unique_ids), batch_size): # Iterate over unique IDs in batches of batch_size
        batch = unique_ids[i:i + batch_size] # Get the current batch of IDs to query using slicing (i to i+batch_size) which creates batches of size batch_size (e.g. 100) until we cover all unique IDs
        query = " OR ".join(f"accession:{uid}" for uid in batch) # Construct the query string for the batch by joining accession:ID for each ID in the batch with OR (e.g. "accession:P00734 OR accession:P12345")
        params = { # Set up the parameters for the API request in a dict
            "query": f"({query})",# The query parameter is set to the constructed query string that searches for the batch of UniProt IDs using the "accession" field. The IDs are combined with "OR" to fetch all relevant entries in one request.
            "format": "fasta", # Request FASTA format for the response
        }
        if include_isoforms: # If we want to include isoforms, add the parameter to the request
            params["includeIsoform"] = "true" # Add the includeIsoform parameter to the request parameters to tell the API to include isoform sequences in the response

        url = "https://rest.uniprot.org/uniprotkb/stream"
        response = session.get(url, params=params) # Make the GET request to the UniProt REST API with the specified URL and parameters using the session for connection pooling

        if response.status_code == 200: # If the request was successful (HTTP status code 200), parse the response
            # Parse multi-FASTA: split on '>' and reconstruct
            current_id = None # Track the current FASTA ID (canonical or isoform) being parsed
            current_lines = [] # Accumulate lines for the current FASTA entry (header + sequence lines)
            for line in response.text.splitlines(): # Iterate over each line in the response text (which is in FASTA format, so it contains header lines starting with '>' and sequence lines)
                if line.startswith(">"): # If the line is a header line (starts with '>'), it indicates the start of a new FASTA entry
                    if current_id and current_lines: # If we were accumulating lines for a previous FASTA entry, save it to results before starting the new one
                        results[current_id] = "\n".join(current_lines) # Save the accumulated lines for the previous FASTA entry under its ID in the results dict. The lines are joined with newline characters to reconstruct the full FASTA entry (header + sequence).
                    # Header format: >sp|P00734|THRB_HUMAN ... or >sp|P00734-2|THRB_HUMAN ...
                    parts = line.split("|") # Split the header line by '|' to extract the ID parts. The expected format is something like ">sp|P00734|THRB_HUMAN ..." where the second part (index 1) is the UniProt ID (which may include isoform suffixes like -2).
                    if len(parts) >= 2: # Check if the header line has the expected format with at least 2 parts when split by '|'. This is a basic validation to ensure we can extract the ID correctly.
                        current_id = parts[1]  # e.g. 'P00734' or 'P00734-2'
                    else:
                        current_id = line[1:].split()[0] # Fallback: take the first word after '>' if the expected format is not met
                    current_lines = [line] # Start accumulating lines for the new FASTA entry, starting with the header line
                else:
                    current_lines.append(line) # If the line is not a header, it is part of the sequence, so we accumulate it in current_lines for the current FASTA entry
            if current_id and current_lines: # After the loop, make sure to save the last accumulated FASTA entry if there is one
                results[current_id] = "\n".join(current_lines) # Save the last FASTA entry to results

            print(f"Batch {i // batch_size + 1}: fetched {len(batch)} IDs") # Print a message indicating how many IDs were fetched in the current batch (using integer division to calculate the batch number)
        else:
            print(f"Batch {i // batch_size + 1} failed with status {response.status_code}: {response.text[:200]}") # If the request failed, print an error message with the status code and the first 200 characters of the response text for debugging

    # Group isoforms under their canonical ID (P00734-2 -> P00734)
    grouped = {} # Store grouped results as {canonical_id: {fasta_id: fasta_string}} where canonical_id is the base ID without isoform suffix (e.g. P00734) and fasta_id can be canonical or isoform (e.g. P00734 or P00734-2)
    for fasta_id, fasta_seq in results.items(): # Iterate over the fetched FASTA entries in results
        canonical = fasta_id.split("-")[0] # Get the canonical ID by splitting the fasta_id on '-' and taking the first part (e.g. P00734-2 -> P00734)
        if canonical not in grouped: # If this canonical ID is not already in the grouped dict, initialize it with an empty dict to hold its FASTA entries (canonical and isoforms)
            grouped[canonical] = {} # Initialize the entry for this canonical ID in the grouped dict
        grouped[canonical][fasta_id] = fasta_seq # Add the FASTA entry to the grouped dict under its canonical ID. This allows us to have all isoforms grouped together under their canonical ID.

    print(f"\nFetched sequences for {len(grouped)} canonical proteins ({len(results)} total including isoforms)") # Print a summary of how many canonical proteins were fetched (length of grouped) and the total number of FASTA entries including isoforms (length of results)
    return grouped

# Run the batch fetch
unique_ids = target_df['uniprot_id'].dropna().unique().tolist() # Get the unique UniProt IDs from the target_df DataFrame, dropping any NaN values and converting to a list for processing
fasta_results = fetch_fasta_batch(unique_ids, include_isoforms=True) # Fetch the FASTA sequences for the unique UniProt IDs, including isoforms, using the batch fetch function defined above. The results will be a dict grouped by canonical ID with their corresponding FASTA entries.


Batch 1: fetched 100 IDs
Batch 2: fetched 100 IDs
Batch 3: fetched 100 IDs
Batch 4: fetched 100 IDs
Batch 5: fetched 100 IDs
Batch 6: fetched 100 IDs
Batch 7: fetched 100 IDs
Batch 8: fetched 100 IDs
Batch 9: fetched 100 IDs
Batch 10: fetched 100 IDs
Batch 11: fetched 9 IDs

Fetched sequences for 1008 canonical proteins (2303 total including isoforms)


In [ ]:

# Step 3: Inspect results
for uid in list(fasta_results.keys())[:3]: # Inspect the first 3 canonical IDs in the fetched results to see how many isoforms they have and print their headers. This is just a sample to verify that the fetching and grouping of isoforms is working correctly.
    isoforms = fasta_results[uid] # Get the dict of FASTA entries for this canonical ID, which includes the canonical entry and any isoforms (e.g. P00734: {'P00734': '...FASTA...', 'P00734-2': '...FASTA...'})
    print(f"{uid}: {len(isoforms)} isoform(s) — {list(isoforms.keys())}") # Print the canonical ID, the number of isoforms (including the canonical entry itself), and the list of FASTA IDs (canonical and isoforms) that were fetched for this protein
    # Print first 80 chars of each isoform header
    for iso_id, fasta in isoforms.items(): # Iterate over each FASTA entry for this canonical ID (including isoforms) to print the first 80 characters of the header line for inspection. This helps verify that we are correctly fetching and grouping the FASTA entries.
        print(f"{fasta.splitlines()[0][:80]}") # Print the first 80 characters of the header line (the first line of the FASTA entry) for this isoform to verify the content. This is useful for checking that we have the correct entries and that isoforms are included as expected.
        print() # print the begging of the sequence line for additional verification
        if len(fasta.splitlines()) > 1: # Check if there are sequence lines after the header to print
            print(f"{fasta.splitlines()[1][:80]}") # Print the first 80 characters of the first sequence line for additional verification. This helps confirm that we are getting valid FASTA entries with both headers and sequences.
    print() 


K9N7C7: 1 isoform(s) — ['K9N7C7']
>sp|K9N7C7|R1AB_MERS1 Replicase polyprotein 1ab OS=Middle East respiratory syndr

MSFVAGVTAQGARGTYRAALNSEKHQDHVSLTVPLCGSGNLVEKLSPWFMDGENAYEVVK

O00469: 3 isoform(s) — ['O00469', 'O00469-2', 'O00469-3']
>sp|O00469|PLOD2_HUMAN Procollagen-lysine,2-oxoglutarate 5-dioxygenase 2 OS=Homo

MGGCTVKPQLLLLALVLHPWNPCLGADSEKPSSIPTDKLLVITVATKESDGFHRFMQSAK
>sp|O00469-2|PLOD2_HUMAN Isoform 2 of Procollagen-lysine,2-oxoglutarate 5-dioxyg

MGGCTVKPQLLLLALVLHPWNPCLGADSEKPSSIPTDKLLVITVATKESDGFHRFMQSAK
>sp|O00469-3|PLOD2_HUMAN Isoform 3 of Procollagen-lysine,2-oxoglutarate 5-dioxyg

MLENHILHKRIYILTFFSQQIFILCHAHFIFFFTVRDFCRQDEKCDYYFSVDADVVLTNP

O00591: 1 isoform(s) — ['O00591']
>sp|O00591|GBRP_HUMAN Gamma-aminobutyric acid receptor subunit pi OS=Homo sapien

MNYSLHLAFVCLSLFTERMCIQGSQFNVEVGRSDKLSLPGFENLTAGYNKFLRPNFGGEP



In [38]:

# Mapping the sequences back to the original DataFrame

def map_fasta_to_df(target_df, fasta_results):
    """Map the fetched FASTA sequences back to the original DataFrame based on UniProt IDs.
    If multiple isoforms exist, concatenate them with '###' separator, canonical first."""
    target_df_w_fasta = target_df.copy()

    def get_all_isoforms(uid):
        """Inner helper: for a single UniProt ID, return (headers, sequences, n_isoforms)."""
        canonical = uid.split("-")[0] # Get the canonical ID by splitting the UniProt ID on '-' and taking the first part (e.g. P00734-2 -> P00734)
        isoform_dict = fasta_results.get(canonical, {}) # Get the dict of FASTA entries for this canonical ID from the fetched results. This will include the canonical entry and any isoforms (e.g. {'P00734': '...FASTA...', 'P00734-2': '...FASTA...'}). If the canonical ID is not found in the results, return an empty dict.
        if not isoform_dict: 
            return None, None, None
        # Sort keys so canonical (no dash) comes first, then isoforms in order
        sorted_ids = sorted(isoform_dict.keys(), key=lambda x: (x != canonical, x)) # Sort the FASTA IDs for this canonical protein so that the canonical entry (which does not have a dash) comes first, followed by any isoforms in sorted order.
        # Extract headers and sequences separately for each isoform
        headers = []
        sequences = []
        for iso_id in sorted_ids:
            fasta = isoform_dict[iso_id]
            lines = fasta.split("\n")
            headers.append(lines[0])
            sequences.append("\n".join(lines[1:]))
        # Join with ### separator
        all_headers = " ### ".join(headers)
        all_sequences = " ### ".join(sequences)
        num_isoforms = len(sorted_ids)
        return all_headers, all_sequences, num_isoforms

    # Apply the helper to each row's uniprot_id
    results = target_df_w_fasta['uniprot_id'].apply(get_all_isoforms) # Apply the helper function to each UniProt ID in the DataFrame
    target_df_w_fasta['fasta_info'] = results.apply(lambda x: x[0])
    target_df_w_fasta['fasta_sequence'] = results.apply(lambda x: x[1])
    target_df_w_fasta['n_isoforms'] = results.apply(lambda x: x[2])

    return target_df_w_fasta


In [40]:
final_df = map_fasta_to_df(target_df, fasta_results)
print(final_df['fasta_info'].iloc[1])
print()
print(final_df['fasta_sequence'].iloc[1])




>sp|P14416|DRD2_HUMAN D(2) dopamine receptor OS=Homo sapiens OX=9606 GN=DRD2 PE=1 SV=2 ### >sp|P14416-2|DRD2_HUMAN Isoform 2 of D(2) dopamine receptor OS=Homo sapiens OX=9606 GN=DRD2 ### >sp|P14416-3|DRD2_HUMAN Isoform 3 of D(2) dopamine receptor OS=Homo sapiens OX=9606 GN=DRD2

MDPLNLSWYDDDLERQNWSRPFNGSDGKADRPHYNYYATLLTLLIAVIVFGNVLVCMAVS
REKALQTTTNYLIVSLAVADLLVATLVMPWVVYLEVVGEWKFSRIHCDIFVTLDVMMCTA
SILNLCAISIDRYTAVAMPMLYNTRYSSKRRVTVMISIVWVLSFTISCPLLFGLNNADQN
ECIIANPAFVVYSSIVSFYVPFIVTLLVYIKIYIVLRRRRKRVNTKRSSRAFRAHLRAPL
KGNCTHPEDMKLCTVIMKSNGSFPVNRRRVEAARRAQELEMEMLSSTSPPERTRYSPIPP
SHHQLTLPDPSHHGLHSTPDSPAKPEKNGHAKDHPKIAKIFEIQTMPNGKTRTSLKTMSR
RKLSQQKEKKATQMLAIVLGVFIICWLPFFITHILNIHCDCNIPPVLYSAFTWLGYVNSA
VNPIIYTTFNIEFRKAFLKILHC ### MDPLNLSWYDDDLERQNWSRPFNGSDGKADRPHYNYYATLLTLLIAVIVFGNVLVCMAVS
REKALQTTTNYLIVSLAVADLLVATLVMPWVVYLEVVGEWKFSRIHCDIFVTLDVMMCTA
SILNLCAISIDRYTAVAMPMLYNTRYSSKRRVTVMISIVWVLSFTISCPLLFGLNNADQN
ECIIANPAFVVYSSIVSFYVPFIVTLLVYIKIYIVLRRRRKRVNTKRSSRAFRAHLRAPL
KEAARRAQELEMEMLSSTSPP

In [41]:
def send_results_to_csv(df, filename, file_path):
    """Save the DataFrame with FASTA info to a CSV file."""
    full_path = f"{file_path}\\{filename}"
    df.to_csv(full_path, index=False)
    print(f"Results saved to {full_path}")


send_results_to_csv(final_df, "mapped_targets_with_fasta.csv", r"C:\Users\ashto\ddi-prediction\data\sample")

Results saved to C:\Users\ashto\ddi-prediction\data\sample\mapped_targets_with_fasta.csv


In [42]:
final_df.head()

,drug,target,uniprot_id,gene,source,fasta_info,fasta_sequence,n_isoforms
0,Bivalirudin,Prothrombin,P00734,F2,Swiss-Prot,>sp|P00734|THRB_HUMAN Prothrombin OS=Homo sapi...,MAHVRGLQLPGCLALAALCSLVHSQHVFLAPQQARSLLQRVRRANT...,1.0
1,Chlorprothixene,D(2) dopamine receptor,P14416,DRD2,Swiss-Prot,>sp|P14416|DRD2_HUMAN D(2) dopamine receptor O...,MDPLNLSWYDDDLERQNWSRPFNGSDGKADRPHYNYYATLLTLLIA...,3.0
2,Chlorprothixene,D(1A) dopamine receptor,P21728,DRD1,Swiss-Prot,>sp|P21728|DRD1_HUMAN D(1A) dopamine receptor ...,MRTLNTSAMDGTGLVVERDFSVRILTACFLSLLILSTLLGNTLVCA...,1.0
3,Chlorprothixene,D(3) dopamine receptor,P35462,DRD3,Swiss-Prot,>sp|P35462|DRD3_HUMAN D(3) dopamine receptor O...,MASLSQLSGHLNYTCGAENSTGASQARPHAYYALSYCALILAIVFG...,2.0
4,Chlorprothixene,5-hydroxytryptamine receptor 2A,P28223,HTR2A,Swiss-Prot,>sp|P28223|5HT2A_HUMAN 5-hydroxytryptamine rec...,MDILCEENTSLSSTTNSLMQLNDDTRLYSNDFNSGEANTSDAFNWT...,2.0


In [ ]:
# Gene ontology information extraction from uniprot
# Using goatools to extact GO terms for the unique UniProt IDs we have
# https://github.com/tanghaibao/goatools - GITHUB REPO FOR GOATOOLS
# https://www.nature.com/articles/s41598-018-28948-z - PAPER ON GOATOOLS

